[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/EN/Seminar_Notebooks/chapter12_seminar_notebook.ipynb)

---

# Chapter 12 Seminar: Spectral Analysis of Time Series

**Course:** Time Series Analysis and Forecasting  
**Program:** Bachelor program, Faculty of Cybernetics, Statistics and Economic Informatics, Bucharest University of Economic Studies, Romania  
**Academic Year:** 2025-2026

---

## Seminar Objectives

In this practical seminar, you will:
1. Understand the time-domain vs frequency-domain duality
2. Compute and interpret the periodogram and its statistical properties
3. Apply smoothed spectral estimators (Welch, multitaper)
4. Estimate parametric spectral densities from fitted AR models
5. Analyse linear filters (HP, BK) in the frequency domain
6. Estimate long-memory parameters using GPH and Local Whittle
7. Perform cross-spectral analysis: coherence, phase, and group delay

## Setup

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install statsmodels yfinance arch -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from scipy.fft import fft, fftfreq
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.tsa.filters.bk_filter import bkfilter
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

COLORS = {
    'blue': '#1A3A6E', 'red': '#DC3545', 'green': '#2E7D32',
    'amber': '#B5853F', 'orange': '#E6802E', 'purple': '#8E44AD',
    'gray': '#666666', 'light_blue': '#5B8BD4'
}

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.facecolor': 'none',
    'figure.facecolor': 'none',
    'savefig.facecolor': 'none',
    'legend.frameon': False
})

print('Setup complete. Libraries loaded.')

---
# Part I: Fundamentals (Year 3 Level)

**Objectives:** Understand time vs frequency domain, compute periodograms, identify dominant cycles, interpret spectra graphically.

**Dataset:** S&P 500 daily returns (via yfinance) and US quarterly real GDP (via statsmodels).

## Exercise 1: Log-Returns and ACF

**Task:** Download S&P 500 data using yfinance, compute log-returns, and plot price, returns, and ACF side-by-side.  
Observe whether returns are serially uncorrelated while squared returns are not — motivating spectral analysis of volatility.

In [ ]:
import yfinance as yf

# Download S&P 500 (use period='10y' for 10 years)
ticker = yf.Ticker('^GSPC')
data = ticker.history(period='10y')
prices = data['Close'].dropna()
returns = np.log(prices / prices.shift(1)).dropna()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: Price
axes[0].plot(prices.index, prices.values, color=COLORS['blue'], linewidth=0.8)
axes[0].set_title('S&P 500 Price Level', fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Price (USD)')

# Panel 2: Log-returns
axes[1].plot(returns.index, returns.values, color=COLORS['red'], linewidth=0.5, alpha=0.7)
axes[1].axhline(0, color=COLORS['gray'], linestyle='--', linewidth=0.8)
axes[1].set_title(r'Log-Returns $r_t = \log(P_t/P_{t-1})$', fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Log-return')

# Panel 3: ACF of returns
acf_vals = acf(returns, nlags=40, alpha=0.05)
lags = np.arange(len(acf_vals[0]))
conf = 1.96 / np.sqrt(len(returns))
axes[2].bar(lags, acf_vals[0], color=COLORS['blue'], alpha=0.6, width=0.5)
axes[2].axhline(conf, color=COLORS['red'], linestyle='--', linewidth=1, label='95% CI')
axes[2].axhline(-conf, color=COLORS['red'], linestyle='--', linewidth=1)
axes[2].set_title('ACF of Log-Returns', fontweight='bold')
axes[2].set_xlabel('Lag')
axes[2].set_ylabel('ACF')
axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, frameon=False)

for ax in axes[:2]:
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'Sample size: {len(returns):,} daily observations')
print(f'Mean return: {returns.mean():.4f} | Std: {returns.std():.4f}')

In [ ]:
# ACF of squared returns (volatility clustering)
sq_returns = returns**2
acf_sq = acf(sq_returns, nlags=40)
conf = 1.96 / np.sqrt(len(sq_returns))

fig, ax = plt.subplots(figsize=(14, 4))
lags = np.arange(len(acf_sq))
ax.bar(lags, acf_sq, color=COLORS['amber'], alpha=0.7, width=0.5)
ax.axhline(conf, color=COLORS['red'], linestyle='--', linewidth=1, label='95% CI')
ax.axhline(-conf, color=COLORS['red'], linestyle='--', linewidth=1)
ax.set_title('ACF of Squared Log-Returns (Volatility Clustering)', fontweight='bold')
ax.set_xlabel('Lag')
ax.set_ylabel('ACF')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)
ax.set_facecolor('none')
fig.patch.set_alpha(0)
plt.show()

print('Observation: Squared returns show strong autocorrelation -> ARCH effects')
print('This motivates spectral analysis of the volatility process')

## Exercise 2: The Raw Periodogram

**Task:** Compute the FFT-based periodogram of S&P 500 log-returns.  
Plot on linear and log scales. Identify where spectral energy is concentrated and whether the spectrum resembles white noise.

In [ ]:
r = returns.values
n = len(r)

# FFT-based periodogram (scipy), fs=252 trading days/year
freqs, psd_raw = signal.periodogram(r, fs=252)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].plot(freqs[1:], psd_raw[1:], color=COLORS['blue'], linewidth=0.5, alpha=0.8)
axes[0].set_title('Periodogram of S&P 500 Log-Returns', fontweight='bold')
axes[0].set_xlabel('Frequency (cycles/year)')
axes[0].set_ylabel('Spectral density')
axes[0].set_facecolor('none')

# Log scale (better for detecting small variations)
axes[1].semilogy(freqs[1:], psd_raw[1:], color=COLORS['red'], linewidth=0.5, alpha=0.8)
axes[1].set_title('Periodogram (Log Scale)', fontweight='bold')
axes[1].set_xlabel('Frequency (cycles/year)')
axes[1].set_ylabel('Log spectral density')
axes[1].set_facecolor('none')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

# Find dominant frequencies
top_idx = np.argsort(psd_raw[1:])[-5:][::-1] + 1
print('Top 5 dominant frequencies:')
for i in top_idx:
    if freqs[i] > 0:
        print(f'  freq = {freqs[i]:.2f} cycles/year  |  period = {1/freqs[i]:.1f} years  |  power = {psd_raw[i]:.6f}')

### Question 2.1

Where is most of the energy concentrated in the S&P 500 return spectrum?  
Is the spectrum "flat" (white noise) or "colored" (structured)?

**YOUR ANSWER HERE**

## Exercise 3: Inconsistency of the Periodogram

**Task:** Simulate an AR(1) process and show that the periodogram does NOT converge to the true spectrum as N grows.  
Then verify that $I(\omega_j) / f(\omega_j) \sim \chi^2_2 / 2$ via Monte Carlo.

In [ ]:
# Simulate AR(1): Y_t = 0.7 Y_{t-1} + eps_t
phi = 0.7
sigma2 = 1.0

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
sample_sizes = [128, 256, 512, 2048]

for ax, N in zip(axes.flatten(), sample_sizes):
    # Simulate
    y = np.zeros(N)
    eps = np.random.randn(N)
    for t in range(1, N):
        y[t] = phi * y[t-1] + eps[t]

    # Periodogram
    freqs_ar, I_omega = signal.periodogram(y)

    # Theoretical spectrum: f(omega) = sigma2/(2*pi) / |1 - phi*exp(-i*omega)|^2
    omega = 2 * np.pi * freqs_ar
    f_theory = (sigma2 / (2 * np.pi)) / np.abs(1 - phi * np.exp(-1j * omega))**2

    ax.plot(freqs_ar[1:], I_omega[1:], color=COLORS['blue'], linewidth=0.4,
            alpha=0.7, label='Periodogram')
    ax.plot(freqs_ar[1:], f_theory[1:], color=COLORS['red'], linewidth=2,
            label='True $f(\\omega)$')
    ax.set_title(f'N = {N:,}', fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.set_ylabel('Spectral density')
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)

fig.suptitle(
    'Periodogram Inconsistency: Variance Does Not Shrink as N \u2192 \u221e',
    fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate: I(omega_j) ~ f(omega_j) * chi2(2)/2
N = 512
phi = 0.7
n_sim = 2000
omega_target = np.pi / 4  # Target frequency

ratios = []
for _ in range(n_sim):
    y = np.zeros(N)
    eps = np.random.randn(N)
    for t in range(1, N):
        y[t] = phi * y[t-1] + eps[t]
    freqs_s, I = signal.periodogram(y)
    idx = np.argmin(np.abs(2 * np.pi * freqs_s - omega_target))
    omega_j = 2 * np.pi * freqs_s[idx]
    f_true = (1.0 / (2 * np.pi)) / np.abs(1 - phi * np.exp(-1j * omega_j))**2
    ratios.append(I[idx] / f_true)

from scipy.stats import chi2
x = np.linspace(0, 10, 200)
chi2_pdf = chi2.pdf(2*x, df=2) * 2  # chi2(2)/2 pdf

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ratios, bins=50, density=True, color=COLORS['blue'], alpha=0.6,
        label=r'$I(\omega_j)/f(\omega_j)$ (simulated)')
ax.plot(x, chi2_pdf, color=COLORS['red'], linewidth=2,
        label=r'$\chi^2_2/2$ density')
ax.set_title(
    r'Distribution of $I(\omega_j)/f(\omega_j)$: confirms $I(\omega_j) \sim f(\omega_j)\cdot\chi^2_2/2$',
    fontweight='bold')
ax.set_xlabel(r'$I(\omega_j)/f(\omega_j)$')
ax.set_ylabel('Density')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
ax.set_facecolor('none')
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'Simulated mean: {np.mean(ratios):.3f}  (chi2(2)/2 mean = 1.000)')
print(f'Simulated var:  {np.var(ratios):.3f}  (chi2(2)/2 variance = 1.000)')

## Exercise 4: AR(1) Spectral Density — Theory vs Estimation

**Task:** Simulate an AR(1) process with $\phi = 0.8$.  
Compare the theoretical spectrum, the raw periodogram, and the Welch estimate on linear and log scales.  
Understand what a *red spectrum* looks like.

In [ ]:
# Simulate Y_t = 0.8 * Y_{t-1} + eps_t
phi = 0.8
sigma2 = 1.0
N = 1000

y = np.zeros(N)
eps = np.random.randn(N)
for t in range(1, N):
    y[t] = phi * y[t-1] + eps[t]

# Theoretical spectrum
omega = np.linspace(0, np.pi, 500)
f_theory = (sigma2 / (2 * np.pi)) / (1 - 2*phi*np.cos(omega) + phi**2)

# Estimated: raw periodogram
freqs_est, I_est = signal.periodogram(y)
omega_est = 2 * np.pi * freqs_est

# Welch estimate
freqs_w, psd_w = signal.welch(y, nperseg=128)
omega_w = 2 * np.pi * freqs_w

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, yscale, title in zip(
    axes, ['linear', 'log'],
    ['AR(1) Spectrum: Linear Scale', 'AR(1) Spectrum: Log Scale']
):
    ax.plot(omega[1:], f_theory[1:], color=COLORS['red'], linewidth=2.5,
            label='Theoretical $f(\\omega)$', zorder=3)
    ax.plot(omega_est[1:], I_est[1:], color=COLORS['blue'], linewidth=0.4,
            alpha=0.5, label='Raw periodogram')
    ax.plot(omega_w[1:], psd_w[1:] / (2*np.pi), color=COLORS['green'], linewidth=1.5,
            label='Welch estimate')
    ax.set_yscale(yscale)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency $\\omega$ (radians/period)')
    ax.set_ylabel('Spectral density')
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)

fig.suptitle(
    f'AR(1) with $\\phi = {phi}$: "Red Spectrum" \u2014 Most Power at Low Frequencies',
    fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'Theoretical peak: omega=0, f(0) = {(sigma2/(2*np.pi))/(1-phi)**2:.3f}')
print(f'Theoretical trough: omega=pi, f(pi) = {(sigma2/(2*np.pi))/(1+phi)**2:.3f}')
print(f'Spectral ratio (low/high freq): {((1+phi)/(1-phi))**2:.1f}x')

---
# Part II: Advanced Estimation (Master Level)

**Objectives:** Compare estimation methods, understand the bias-variance tradeoff in spectral estimation, and analyse linear filters in the frequency domain.

## Exercise 5: Welch vs Multitaper — Bias-Variance Tradeoff

**Task:** Apply three spectral estimators to the sunspot series (annual data, 1700-present).  
Compare the raw periodogram, Welch's method, and the multitaper DPSS estimator.  
Identify the well-known 11-year solar cycle.

In [ ]:
# Load sunspot data
sunspots = sm.datasets.sunspots.load_pandas().data
y_sun = sunspots['SUNACTIVITY'].values
n_sun = len(y_sun)

# 1. Raw periodogram
freqs_raw, I_raw = signal.periodogram(y_sun)

# 2. Welch
freqs_w, I_w = signal.welch(y_sun, nperseg=64)

# 3. Multitaper (manual DPSS)
from scipy.signal.windows import dpss
NW = 4
K = 7
tapers, _ = dpss(n_sun, NW, K, return_ratios=True)
mt_spectra = []
for taper in tapers:
    f_mt, I_mt_k = signal.periodogram(y_sun * taper)
    mt_spectra.append(I_mt_k)
I_mt = np.mean(mt_spectra, axis=0)
freqs_mt = f_mt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titles = [
    'Raw Periodogram\n(Inconsistent)',
    "Welch's Method\n(Segment averaging, nperseg=64)",
    'Multitaper (DPSS)\n(NW=4, K=7 tapers)'
]
data_pairs = [
    (freqs_raw[1:], I_raw[1:], COLORS['blue']),
    (freqs_w[1:], I_w[1:], COLORS['green']),
    (freqs_mt[1:], I_mt[1:], COLORS['purple'])
]

for ax, (fr, spec, col), title in zip(axes, data_pairs, titles):
    ax.semilogy(fr, spec, color=col, linewidth=0.8, alpha=0.9)
    f_11 = 1/11
    ax.axvline(f_11, color=COLORS['red'], linestyle='--', linewidth=1.5, label='11-yr cycle')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency (cycles/year)')
    ax.set_ylabel('Log spectral density')
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, frameon=False)

fig.suptitle('Sunspot Data: Comparison of Spectral Estimators', fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison: coefficient of variation near 11-yr frequency
f_target = 1/11
window = 0.02  # bandwidth around 11-yr cycle

def spectral_variance(freqs, spec, f_center, bw):
    mask = (freqs > f_center - bw) & (freqs < f_center + bw)
    vals = spec[mask]
    return vals.std() / vals.mean() if len(vals) > 1 else np.nan

print('Estimator comparison near 11-year cycle (f \u2248 0.091):')
print('=' * 55)
for name, (fr, spec, _) in zip(['Raw', 'Welch', 'Multitaper'], data_pairs):
    cv = spectral_variance(fr, spec, f_target, window)
    print(f'  {name:12s}: CoV = {cv:.3f}  (lower = less variance)')

## Exercise 6: Parametric AR Spectral Estimation

**Task:** Use AIC to select the optimal AR order for the sunspot series, then compute the implied spectral density.  
Compare the parametric AR spectrum against the nonparametric multitaper estimate.

In [ ]:
from statsmodels.tsa.ar_model import AutoReg

# Select AR order by AIC
aic_vals = {}
for p in range(1, 20):
    try:
        mod = AutoReg(y_sun, lags=p, old_names=False).fit()
        aic_vals[p] = mod.aic
    except Exception:
        pass
p_opt = min(aic_vals, key=aic_vals.get)
print(f'Optimal AR order by AIC: p = {p_opt}')

# Fit optimal AR
mod_opt = AutoReg(y_sun, lags=p_opt, old_names=False).fit()
ar_params = mod_opt.params[1:]  # exclude intercept
sigma2_ar = mod_opt.sigma2

# Compute AR spectral density
omega_ar = np.linspace(0, np.pi, 1000)
ar_z = np.array([
    np.sum([ar_params[k] * np.exp(-1j*(k+1)*w) for k in range(len(ar_params))])
    for w in omega_ar
])
f_ar = (sigma2_ar / (2*np.pi)) / np.abs(1 - ar_z)**2

fig, ax = plt.subplots(figsize=(14, 5))
ax.semilogy(freqs_mt[1:], I_mt[1:], color=COLORS['purple'], linewidth=1,
            alpha=0.8, label='Multitaper')
ax.semilogy(omega_ar[1:] / (2*np.pi), f_ar[1:], color=COLORS['red'], linewidth=2.5,
            label=f'AR({p_opt}) parametric')
ax.axvline(1/11, color=COLORS['amber'], linestyle='--', linewidth=1.5, label='11-yr cycle')
ax.set_title(
    f'Sunspot Spectrum: Multitaper vs AR({p_opt}) Parametric Estimate',
    fontweight='bold')
ax.set_xlabel('Frequency (cycles/year)')
ax.set_ylabel('Log spectral density')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
ax.set_facecolor('none')
fig.patch.set_alpha(0)
plt.show()

print(f'\nAR({p_opt}) coefficients: {ar_params.round(3)}')
print(f'Residual variance: {sigma2_ar:.4f}')

## Exercise 7: HP Filter in the Frequency Domain

**Task:** Derive and plot the HP filter transfer function for $\lambda = 1600$ (quarterly data).  
Apply the HP and BK filters to US real GDP and compare the extracted business cycles.

In [ ]:
# Load US quarterly real GDP
gdp_data = sm.datasets.macrodata.load_pandas().data
gdp = np.log(gdp_data['realgdp'].values)
n_gdp = len(gdp)

# Apply HP filter (lambda=1600 for quarterly data)
lam = 1600
gdp_cycle, gdp_trend = hpfilter(gdp, lamb=lam)

# Theoretical HP transfer function
omega = np.linspace(0.001, np.pi, 1000)
# HP cycle gain: g(omega) = lambda*[2(1-cos(omega))]^2 / (1 + lambda*[2(1-cos(omega))]^2)
hp_gain_sq = (lam * (2*(1-np.cos(omega)))**2) / (1 + lam*(2*(1-np.cos(omega)))**2)

# BK filter (6-32 quarters)
gdp_bk = bkfilter(gdp, low=6, high=32, K=12)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Transfer function
axes[0].plot(omega/np.pi, hp_gain_sq, color=COLORS['blue'], linewidth=2,
             label='HP filter gain $|A_{HP}(\\omega)|^2$')
axes[0].axvline(2/32, color=COLORS['green'], linestyle='--', linewidth=1.5,
                label='32-qtr lower bound')
axes[0].axvline(2/6, color=COLORS['amber'], linestyle='--', linewidth=1.5,
                label='6-qtr upper bound')
axes[0].set_title('HP Filter Transfer Function ($\\lambda=1600$)', fontweight='bold')
axes[0].set_xlabel('Frequency (fraction of $\\pi$)')
axes[0].set_ylabel('Gain$^2$')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, frameon=False)
axes[0].set_facecolor('none')

# Panel 2: GDP and trend
t = np.arange(n_gdp)
axes[1].plot(t, gdp, color=COLORS['blue'], linewidth=1, label='Log Real GDP')
axes[1].plot(t, gdp_trend, color=COLORS['red'], linewidth=2, label='HP Trend')
axes[1].set_title('US Real GDP: Level and HP Trend', fontweight='bold')
axes[1].set_xlabel('Quarter')
axes[1].set_ylabel('Log GDP')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
axes[1].set_facecolor('none')

# Panel 3: Extracted cycle
t_bk = np.arange(len(gdp_bk))
axes[2].plot(t[12:-12], gdp_cycle[12:-12], color=COLORS['purple'], linewidth=1.2,
             label='HP cycle', alpha=0.8)
axes[2].plot(t_bk, gdp_bk, color=COLORS['orange'], linewidth=1.5,
             label='BK cycle (6-32q)', alpha=0.8)
axes[2].axhline(0, color=COLORS['gray'], linewidth=0.8, linestyle='--')
axes[2].set_title('Business Cycle: HP vs BK Filter', fontweight='bold')
axes[2].set_xlabel('Quarter')
axes[2].set_ylabel('Cycle component')
axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
axes[2].set_facecolor('none')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('\nHP Filter properties:')
print(f'  Gain at omega=0 (trend): {0:.3f} (removes all trend)')
print(f'  Gain at omega=pi (high freq): {hp_gain_sq[-1]:.3f}')
print(f'  Half-power frequency: omega \u2248 {omega[np.argmin(np.abs(hp_gain_sq-0.5))]/np.pi:.3f}*pi')

---
# Part III: PhD Extension

**Objectives:** Long memory estimation (GPH, Local Whittle), cross-spectral analysis with phase interpretation.

## Exercise 8: Long Memory — GPH Log-Periodogram Regression

**Task:** Implement the Geweke-Porter-Hudak (GPH) estimator.  
Simulate ARFIMA(0, d, 0) processes for $d \in \{0, 0.3, 0.45\}$ and verify that GPH recovers the true $d$ via the log-periodogram regression.

In [ ]:
def gph_estimator(y, m_fraction=0.5):
    """GPH log-periodogram regression estimator of d."""
    n = len(y)
    freqs_g, I = signal.periodogram(y)
    m = int(np.floor(n**m_fraction))
    j = np.arange(1, m+1)
    omega_j = 2 * np.pi * j / n
    log_I = np.log(I[j])
    log_4sin2 = np.log(4 * np.sin(omega_j/2)**2)
    # OLS: log I(omega_j) = const - d * log(4sin^2(omega_j/2)) + error
    X = np.column_stack([np.ones(m), -log_4sin2])
    beta = np.linalg.lstsq(X, log_I, rcond=None)[0]
    d_hat = beta[1]
    se = np.sqrt(np.pi**2 / (6 * m) / (np.var(-log_4sin2)))
    return d_hat, se, m


def simulate_arfima(n, d, sigma=1.0):
    """Simulate ARFIMA(0,d,0) using truncated MA representation."""
    p_approx = min(n//4, 500)
    pi = np.zeros(p_approx)
    pi[0] = -d
    for k in range(1, p_approx):
        pi[k] = pi[k-1] * (k - 1 - d) / (k + 1)
    eps = np.random.randn(n + p_approx) * sigma
    y_out = np.zeros(n + p_approx)
    for t in range(p_approx, n + p_approx):
        y_out[t] = eps[t] - np.dot(
            pi[:min(t, p_approx)],
            y_out[t-1:t-p_approx-1:-1][:p_approx]
        )
    return y_out[p_approx:]


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
d_values = [0.0, 0.3, 0.45]
labels = ['d=0 (ARMA, short memory)', 'd=0.3 (long memory)', 'd=0.45 (strong long memory)']
colors_gph = [COLORS['blue'], COLORS['amber'], COLORS['red']]
N = 1000

for ax, d_true, label, col in zip(axes, d_values, labels, colors_gph):
    y = simulate_arfima(N, d_true)
    freqs_g, I_g = signal.periodogram(y)
    d_hat, se, m = gph_estimator(y)
    j_idx = np.arange(1, m+1)
    omega_j = 2 * np.pi * j_idx / N
    log_4sin2 = np.log(4 * np.sin(omega_j/2)**2)
    log_I = np.log(I_g[j_idx])
    ax.scatter(-log_4sin2, log_I, color=col, s=8, alpha=0.6, label='log periodogram')
    x_line = np.linspace(-log_4sin2.max(), -log_4sin2.min(), 100)
    ax.plot(
        x_line,
        d_hat * x_line + (log_I.mean() - d_hat*(-log_4sin2).mean()),
        color='black', linewidth=2,
        label=f'GPH: $\\hat{{d}}$={d_hat:.3f}\u00b1{se:.3f}'
    )
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel(r'$-\log(4\sin^2(\omega_j/2))$')
    ax.set_ylabel(r'$\log I(\omega_j)$')
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1,
              frameon=False, fontsize=9)

fig.suptitle('GPH Estimator: Log-Periodogram Regression for Long Memory',
             fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

## Exercise 9: Local Whittle Estimator — Comparison with GPH

**Task:** Implement the semiparametric Local Whittle estimator.  
Run a Monte Carlo with N=2000, n\_sim=200 to compare bias and RMSE of GPH vs Local Whittle across four values of $d$.

In [ ]:
def local_whittle(y, m_fraction=0.65):
    """Local Whittle estimator of d (semiparametric)."""
    n = len(y)
    m = int(np.floor(n**m_fraction))
    freqs_lw, I_lw = signal.periodogram(y)
    j = np.arange(1, m+1)
    omega_j = 2 * np.pi * j / n
    log_4sin2 = np.log(4 * np.sin(omega_j/2)**2)
    I_j = I_lw[j]

    from scipy.optimize import minimize_scalar

    def objective(d):
        g_j = np.exp(-d * log_4sin2)
        return np.log(np.mean(I_j / g_j)) - d * np.mean(log_4sin2)

    result = minimize_scalar(objective, bounds=(-0.49, 0.99), method='bounded')
    d_lw = result.x
    se_lw = 1 / (2 * np.sqrt(m))
    return d_lw, se_lw


# Monte Carlo
N = 2000
n_sim = 200
d_grid = [0.0, 0.15, 0.30, 0.45]

mc_results = {d: {'gph': [], 'lw': []} for d in d_grid}

print('Running Monte Carlo simulations...')
for d_true in d_grid:
    for _ in range(n_sim):
        y = simulate_arfima(N, d_true)
        d_gph, _, _ = gph_estimator(y, m_fraction=0.5)
        d_lw, _ = local_whittle(y, m_fraction=0.65)
        mc_results[d_true]['gph'].append(d_gph)
        mc_results[d_true]['lw'].append(d_lw)
    print(f'  d = {d_true}: done')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, method, color in zip(axes, ['gph', 'lw'], [COLORS['blue'], COLORS['red']]):
    d_true_vals = np.array(d_grid)
    means = np.array([np.mean(mc_results[d][method]) for d in d_grid])
    stds = np.array([np.std(mc_results[d][method]) for d in d_grid])
    method_label = 'GPH' if method == 'gph' else 'Local Whittle'
    ax.plot(d_true_vals, d_true_vals, 'k--', linewidth=1.5, label='45\u00b0 line (perfect)')
    ax.errorbar(d_true_vals, means, yerr=2*stds, fmt='o', color=color,
                capsize=5, linewidth=2, markersize=8,
                label=f'{method_label} (mean \u00b1 2SD)')
    ax.set_title(
        f'{method_label}: Bias and Variance\n(N={N}, {n_sim} simulations)',
        fontweight='bold')
    ax.set_xlabel('True $d$')
    ax.set_ylabel('Estimated $\\hat{d}$')
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)

fig.suptitle('GPH vs Local Whittle: Monte Carlo Comparison', fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'\n{"d_true":>8} {"GPH_bias":>10} {"LW_bias":>10} {"GPH_RMSE":>10} {"LW_RMSE":>10}')
for d_true in d_grid:
    gph_arr = np.array(mc_results[d_true]['gph'])
    lw_arr = np.array(mc_results[d_true]['lw'])
    print(
        f'{d_true:>8.2f} {np.mean(gph_arr)-d_true:>10.4f} {np.mean(lw_arr)-d_true:>10.4f} '
        f'{np.sqrt(np.mean((gph_arr-d_true)**2)):>10.4f} '
        f'{np.sqrt(np.mean((lw_arr-d_true)**2)):>10.4f}'
    )

## Exercise 10: Cross-Spectral Analysis — Coherence and Phase

**Task:** Simulate a pure lag-4 relationship $Y_t = X_{t-4} + \varepsilon_t$ where $X_t$ is an AR(2) process.  
Estimate squared coherence, the phase spectrum, and group delay. Verify that the group delay recovers the true lag of 4 periods.

In [ ]:
# Simulate Y_t = X_{t-4} + eps_t (pure lag-4 relationship)
N = 1024
t = np.arange(N)

# X_t: AR(2) process (business cycle-like)
X = np.zeros(N)
eps_x = np.random.randn(N)
for i in range(2, N):
    X[i] = 1.3*X[i-1] - 0.7*X[i-2] + eps_x[i]

# Y_t = X_{t-4} + noise
eps_y = 0.5 * np.random.randn(N)
Y = np.zeros(N)
Y[4:] = X[:-4] + eps_y[4:]

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(t[:200], X[:200], color=COLORS['blue'], linewidth=1, label='$X_t$ (leading)')
axes[0].plot(t[:200], Y[:200], color=COLORS['red'], linewidth=1, alpha=0.8,
             label='$Y_t = X_{t-4} + \\varepsilon_t$ (lagging)')
axes[0].set_title('Simulated Series: $Y_t$ lags $X_t$ by 4 periods', fontweight='bold')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Value')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

axes[1].xcorr(X, Y, maxlags=20, color=COLORS['purple'], usevlines=True)
axes[1].axvline(-4, color=COLORS['red'], linestyle='--', linewidth=2, label='True lag = 4')
axes[1].set_title('Cross-Correlation Function: Peak at lag 4', fontweight='bold')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('Cross-correlation')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, frameon=False)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

In [ ]:
# Compute cross-spectral quantities
freqs_coh, Cxy = signal.coherence(X, Y, nperseg=256)
freqs_csd, Pxy = signal.csd(X, Y, nperseg=256)
freqs_xx, Pxx = signal.welch(X, nperseg=256)

# Phase spectrum and group delay
phase = np.angle(Pxy)
group_delay = -np.diff(np.unwrap(phase)) / np.diff(2*np.pi*freqs_csd)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Squared coherence
axes[0].plot(freqs_coh, Cxy, color=COLORS['blue'], linewidth=1.5)
axes[0].axhline(0.5, color=COLORS['gray'], linestyle=':', linewidth=1)
axes[0].fill_between(freqs_coh, 0, Cxy, where=(Cxy > 0.5),
                     color=COLORS['blue'], alpha=0.2, label='High coherence')
axes[0].set_title('Squared Coherence $\\hat{C}^2_{XY}(\\omega)$', fontweight='bold')
axes[0].set_xlabel('Frequency (cycles/period)')
axes[0].set_ylabel('Coherence$^2$')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

# Panel 2: Phase spectrum with theoretical overlay
axes[1].plot(freqs_csd, np.unwrap(phase), color=COLORS['red'], linewidth=1.5,
             label='Estimated phase')
axes[1].plot(freqs_csd, -4*2*np.pi*freqs_csd, color=COLORS['blue'], linewidth=2,
             linestyle='--', label='Theoretical: $\\phi(\\omega)=-4\\omega$')
axes[1].set_title('Phase Spectrum $\\hat{\\phi}_{XY}(\\omega)$', fontweight='bold')
axes[1].set_xlabel('Frequency (cycles/period)')
axes[1].set_ylabel('Phase (radians)')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

# Panel 3: Group delay
axes[2].plot(freqs_csd[:-1], group_delay / (2*np.pi), color=COLORS['purple'], linewidth=1.5,
             label='Estimated group delay')
axes[2].axhline(4, color=COLORS['red'], linestyle='--', linewidth=2, label='True lag = 4 periods')
axes[2].set_title('Group Delay $= -d\\phi/d\\omega$', fontweight='bold')
axes[2].set_xlabel('Frequency (cycles/period)')
axes[2].set_ylabel('Lag (periods)')
axes[2].set_ylim(-2, 10)
axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
axes[2].spines[['top', 'right']].set_visible(False)
axes[2].set_facecolor('none')

fig.suptitle('Cross-Spectral Analysis: $Y_t = X_{t-4} + \\varepsilon_t$',
             fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Cross-spectral analysis results:')
print(f'  Mean squared coherence: {Cxy.mean():.3f}')
print(f'  Freq with highest coherence: {freqs_coh[np.argmax(Cxy)]:.3f}')
mask_pos = freqs_csd[:-1] > 0.01
print(f'  Mean group delay (periods): '
      f'{np.median(group_delay[mask_pos]/(2*np.pi)):.2f}  (true = 4)')

---
## Practice Problems

### Problem 1: Spectral Density of MA(1)

For $Y_t = \varepsilon_t + \theta\varepsilon_{t-1}$ with $\theta = -0.8$:
- Write the theoretical spectral density $f(\omega)$
- At which frequency is the spectral density maximized?
- Is this a "blue" or "red" spectrum?

In [ ]:
theta = -0.8
omega = np.linspace(0, np.pi, 500)
f_ma1 = (1/(2*np.pi)) * (1 + 2*theta*np.cos(omega) + theta**2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(omega/np.pi, f_ma1, color=COLORS['blue'], linewidth=2)
ax.axvline(omega[np.argmax(f_ma1)]/np.pi, color=COLORS['red'], linestyle='--',
           label=f'Peak at $\\omega={omega[np.argmax(f_ma1)]/np.pi:.2f}\\pi$')
ax.set_xlabel('Frequency (fraction of $\\pi$)')
ax.set_ylabel('Spectral density')
ax.set_title(f'MA(1) Spectrum with $\\theta={theta}$', fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)
ax.set_facecolor('none')
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'MA(1) spectral density: f(omega) = (1/2pi)(1 + 2*theta*cos(omega) + theta^2)')
print(f'Maximum at omega = {omega[np.argmax(f_ma1)]:.4f} rad = {omega[np.argmax(f_ma1)]/np.pi:.2f}*pi')
print(f'Spectrum type: {"Blue (high-frequency dominated)" if theta < 0 else "Red (low-frequency dominated)"}')

### Problem 2: Nyquist Frequency

A quarterly time series (4 observations per year) is available.  
- What is the Nyquist frequency in cycles/year?
- What is the shortest cycle that can be detected?
- If a researcher claims to detect an 18-month (1.5-year) cycle, is this possible?

In [ ]:
# Quarterly data: sampling frequency = 4 cycles/year
fs_quarterly = 4  # samples per year

nyquist_freq = fs_quarterly / 2
shortest_period = 1 / nyquist_freq  # years
shortest_period_months = shortest_period * 12

print('Problem 2 Solution: Nyquist Frequency for Quarterly Data')
print('=' * 55)
print(f'Sampling frequency:         {fs_quarterly} observations/year')
print(f'Nyquist frequency:          {nyquist_freq} cycles/year')
print(f'Shortest detectable period: {shortest_period} year = {shortest_period_months:.0f} months')
print()
print(f'18-month cycle: {18/12:.2f} years -> frequency = {12/18:.4f} cycles/year')
print(f'Is {12/18:.4f} < Nyquist ({nyquist_freq})? '
      f'{"YES: detectable" if 12/18 < nyquist_freq else "NO: aliased"}')

freq_grid = np.linspace(0, nyquist_freq, 300)
fig, ax = plt.subplots(figsize=(10, 3))
ax.axvspan(0, nyquist_freq, alpha=0.1, color=COLORS['green'], label='Detectable range')
ax.axvline(nyquist_freq, color=COLORS['red'], linewidth=2, linestyle='--',
           label=f'Nyquist = {nyquist_freq} cycles/yr')
ax.axvline(12/18, color=COLORS['amber'], linewidth=2,
           label=f'18-month cycle ({12/18:.2f} cycles/yr)')
ax.set_xlim(0, nyquist_freq + 0.5)
ax.set_xlabel('Frequency (cycles/year)')
ax.set_title('Quarterly Data: Detectable Frequency Range', fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False)
ax.set_facecolor('none')
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

### Problem 3: GPH Bandwidth Selection

For the GPH estimator applied to a series of length $N = 1000$:  
- Compute the bandwidth $m$ for $m$-fractions of $0.4$, $0.5$, $0.6$, and $0.7$.
- How does the choice of $m$ affect the bias-variance tradeoff?
- Which fraction is typically recommended in the literature?

In [ ]:
N_ex = 1000
m_fractions = [0.4, 0.5, 0.6, 0.7]

print('Problem 3 Solution: GPH Bandwidth Selection')
print('=' * 60)
print(f'{"m-fraction":>12} {"m (# freqs)":>12} {"Asymp. SE":>12} {"Tradeoff":>20}')
print('-' * 60)
for alpha in m_fractions:
    m = int(np.floor(N_ex**alpha))
    se_approx = np.pi / np.sqrt(6 * m)
    if alpha <= 0.45:
        tradeoff = 'Low bias, high var'
    elif alpha <= 0.55:
        tradeoff = 'Balanced (recommended)'
    else:
        tradeoff = 'Low var, higher bias'
    print(f'{alpha:>12.1f} {m:>12d} {se_approx:>12.4f} {tradeoff:>20}')

print()
print('Literature recommendation: m-fraction = 0.5 (Robinson 1995)')
print('Optimal rate under MSE: m ~ N^(4/5) when d is the only unknown')

alphas = np.linspace(0.3, 0.8, 100)
ms = np.floor(N_ex**alphas).astype(int)
ses = np.pi / np.sqrt(6 * ms)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(alphas, ses, color=COLORS['blue'], linewidth=2)
for alpha in m_fractions:
    m = int(np.floor(N_ex**alpha))
    se = np.pi / np.sqrt(6 * m)
    ax.scatter([alpha], [se], s=80, zorder=5,
               color=COLORS['red'] if alpha == 0.5 else COLORS['amber'])
ax.axvline(0.5, color=COLORS['red'], linestyle='--', linewidth=1.5,
           label='m-fraction = 0.5 (recommended)')
ax.set_xlabel('m-fraction ($\\alpha$ where $m = N^\\alpha$)')
ax.set_ylabel('Approx. SE of $\\hat{d}$')
ax.set_title(f'GPH Standard Error vs Bandwidth (N = {N_ex})', fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)
ax.set_facecolor('none')
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

---
## Summary: Chapter 12 Seminar

| Part | Topic | Key Concepts |
|------|-------|--------------|
| I \u2014 Year 3 | Periodogram | FFT, inconsistency, $\chi^2_2$ distribution, red spectrum |
| II \u2014 Master | Estimation | Welch, multitaper, AR parametric, HP transfer function |
| III \u2014 PhD | Long memory & cross-spectral | GPH, Local Whittle, coherence, phase, group delay |

### Key Takeaways

- **Periodogram**: unbiased but inconsistent \u2014 variance $\approx f(\omega)^2$ regardless of $N$; smoothing is essential
- **Welch vs Multitaper**: both reduce variance at the cost of resolution; multitaper offers better spectral concentration control via DPSS tapers
- **AR parametric spectrum**: high resolution for short series; the AIC-selected AR order determines spectral smoothness
- **Long memory**: $d \in (0, 0.5)$ gives slowly decaying ACF and spectral divergence at the origin; GPH is consistent but Local Whittle achieves lower RMSE
- **Phase and group delay**: the slope of the phase spectrum gives the frequency-dependent lag between two series; group delay $= -d\phi/d\omega$
- **HP filter**: acts as a high-pass filter with a $\lambda$-dependent cutoff; use with care \u2014 see Hamilton (2018) for limitations